In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Keepalive (run this FIRST, in its own cell)
# ══════════════════════════════════════════════════════════════════════════════

%%javascript
function ClickConnect(){
    console.log("Keeping Kaggle alive...");
    document.querySelector(".run-button").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
!pip install weaviate-client sentence-transformers tqdm --quiet

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Imports
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import weaviate
import weaviate.classes as wvc
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Config
#
# WEAVIATE SETUP (free, takes 2 minutes):
#   1. Go to https://console.weaviate.cloud
#   2. Sign up (free)
#   3. Create cluster → choose free sandbox → give it any name
#   4. Copy the Cluster URL and API Key into the variables below
# ══════════════════════════════════════════════════════════════════════════════

#CSV_PATH        = "/kaggle/input/your-dataset-name/jira_train.csv"   # ← update
CSV_PATH        = "/kaggle/input/your-dataset-name/jira_demo.csv"   # ← update
WEAVIATE_URL    = "https://your-cluster.weaviate.network"             # ← update
WEAVIATE_KEY    = "your-api-key"                                      # ← update

COLLECTION_NAME = "JiraIssues"
BATCH_SIZE      = 512

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Load CSV
# ══════════════════════════════════════════════════════════════════════════════

print("Loading jira_demo.csv ...")
df = pd.read_csv(CSV_PATH)
print(f"  Total rows : {len(df):,}")

df = df[df["embedding_text"].notna() & (df["embedding_text"].str.strip() != "")]
print(f"  Rows with embedding text : {len(df):,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Load model
# ══════════════════════════════════════════════════════════════════════════════

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
#model = SentenceTransformer("all-MiniLM-L6-v2").to(device)
#model = SentenceTransformer("BAAI/bge-base-en-v1.5").to(device)
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True          # required for nomic
).to(device)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Connect to Weaviate and create collection
# ══════════════════════════════════════════════════════════════════════════════

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_KEY)
)
print(f"Connected to Weaviate: {client.is_ready()}")

# Create collection if it doesn't exist
if not client.collections.exists(COLLECTION_NAME):
    client.collections.create(
        name=COLLECTION_NAME,
        # We supply our own vectors — no built-in vectorizer
        vectorizer_config=wvc.config.Configure.Vectorizer.none(),
        properties=[
            wvc.config.Property(
                name="jira_id",
                data_type=wvc.config.DataType.INT,
                description="Original numeric ID from jira_train.csv"
            ),
            wvc.config.Property(
                name="key",
                data_type=wvc.config.DataType.TEXT,
                description="JIRA ticket key e.g. PDFBOX-4071"
            ),
            wvc.config.Property(
                name="summary",
                data_type=wvc.config.DataType.TEXT,
                description="Bug title"
            ),
            wvc.config.Property(
                name="priority",
                data_type=wvc.config.DataType.TEXT,
                description="Ticket priority"
            ),
            wvc.config.Property(
                name="resolution",
                data_type=wvc.config.DataType.TEXT,
                description="Resolution name"
            ),
            wvc.config.Property(
                name="discussion",
                data_type=wvc.config.DataType.TEXT,
                description="Cleaned comments text"
            ),
        ]
    )
    print(f"Created collection '{COLLECTION_NAME}'")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists — will resume from checkpoint")

collection = client.collections.get(COLLECTION_NAME)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Checkpoint: skip already embedded rows
#           If Kaggle disconnects, re-run from CELL 8 onwards.
#           It checks how many are already in Weaviate and skips them.
# ══════════════════════════════════════════════════════════════════════════════

existing_count = collection.aggregate.over_all(total_count=True).total_count
print(f"Already in Weaviate : {existing_count:,}")
print(f"Total to embed      : {len(df):,}")

# Skip rows already uploaded — order is preserved since we process sequentially
df_remaining = df.iloc[existing_count:].copy().reset_index(drop=True)
print(f"Remaining           : {len(df_remaining):,}")



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Embed and upload in batches
# ══════════════════════════════════════════════════════════════════════════════

if len(df_remaining) == 0:
    print("Nothing left to embed — already complete!")
else:
    print(f"\nEmbedding {len(df_remaining):,} rows in batches of {BATCH_SIZE} ...")

    with collection.batch.fixed_size(batch_size=BATCH_SIZE) as batch:
        for i in tqdm(range(0, len(df_remaining), BATCH_SIZE), desc="Embedding"):
            chunk   = df_remaining.iloc[i : i + BATCH_SIZE]
            # Nomic requires task prefix on documents at index time
            texts   = chunk["embedding_text"].tolist()
            texts   = ["search_document: " + t for t in texts]  # add this line
            vectors = model.encode(
                texts,
                batch_size=BATCH_SIZE,
                device=str(device),
                show_progress_bar=False,
                convert_to_numpy=True,
            )

            for j, (_, row) in enumerate(chunk.iterrows()):
                batch.add_object(
                    properties={
                        "jira_id"   : int(row["id"]),
                        "key"       : str(row["key"]),
                        "summary"   : str(row.get("summary", "")),
                        "priority"  : str(row.get("priority.name", "")),
                        "resolution": str(row.get("resolution.name", "")),
                        "discussion": str(row.get("comments_text", ""))[:2000],
                    },
                    vector=vectors[j].tolist()
                )

    final_count = collection.aggregate.over_all(total_count=True).total_count
    print(f"\nDone. Total in Weaviate: {final_count:,}")

client.close()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — Sanity check query (run after embedding is complete)
#            Tests that retrieval actually works before you close the notebook
# ══════════════════════════════════════════════════════════════════════════════

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_KEY)
)

collection   = client.collections.get(COLLECTION_NAME)
test_query   = "NullPointerException when processing HTTP request on Linux"
test_vector  = model.encode(["search_query: " + test_query], convert_to_numpy=True)[0].tolist()

results = collection.query.near_vector(
    near_vector=test_vector,
    limit=5,
    return_properties=["key", "summary", "priority", "resolution"]
)

print(f"\nTest query: '{test_query}'")
print(f"Top 5 results:")
for i, obj in enumerate(results.objects, 1):
    p = obj.properties
    print(f"  {i}. [{p['key']}] {p['summary'][:80]}")
    print(f"     Priority: {p['priority']} | Resolution: {p['resolution']}")

client.close()